In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from pprint import pprint

from sklearn.metrics import roc_auc_score

In [2]:
train_sampled_df = pd.read_csv('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/train_sampled_new.csv')
train_sampled_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1028811 entries, 0 to 1028810
Data columns (total 20 columns):
 #   Column                                      Non-Null Count    Dtype  
---  ------                                      --------------    -----  
 0   SeriesUID                                   1028811 non-null  object 
 1   InstanceUID                                 1028811 non-null  object 
 2   Modality                                    1028811 non-null  object 
 3   InstanceNumber                              1028811 non-null  int64  
 4   RescaleSlope                                1028811 non-null  float64
 5   RescaleIntercept                            1028811 non-null  float64
 6   left_infraclinoid_internal_carotid_artery   1028811 non-null  int64  
 7   right_infraclinoid_internal_carotid_artery  1028811 non-null  int64  
 8   left_supraclinoid_internal_carotid_artery   1028811 non-null  int64  
 9   right_supraclinoid_internal_carotid_artery  1028811 non-n

In [3]:
train_sampled_df.head()

,SeriesUID,InstanceUID,Modality,InstanceNumber,RescaleSlope,RescaleIntercept,left_infraclinoid_internal_carotid_artery,right_infraclinoid_internal_carotid_artery,left_supraclinoid_internal_carotid_artery,right_supraclinoid_internal_carotid_artery,left_middle_cerebral_artery,right_middle_cerebral_artery,anterior_communicating_artery,left_anterior_cerebral_artery,right_anterior_cerebral_artery,left_posterior_communicating_artery,right_posterior_communicating_artery,basilar_tip,other_posterior_circulation,aneurysm_present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.56949904638593632206...,MRA,1,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.12396711188070994245...,MRA,2,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.27571397853195038984...,MRA,3,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.60143101667068651693...,MRA,4,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.45662927574100362473...,MRA,5,-100.0,-100.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
train_sampled_df['image_file'] = train_sampled_df.apply(lambda row: f'{row.SeriesUID}_I_{row.InstanceNumber}', axis=1)
filename_to_index = dict(zip(train_sampled_df.image_file, train_sampled_df.index))
train_sampled_df

,SeriesUID,InstanceUID,Modality,InstanceNumber,RescaleSlope,RescaleIntercept,left_infraclinoid_internal_carotid_artery,right_infraclinoid_internal_carotid_artery,left_supraclinoid_internal_carotid_artery,right_supraclinoid_internal_carotid_artery,...,right_middle_cerebral_artery,anterior_communicating_artery,left_anterior_cerebral_artery,right_anterior_cerebral_artery,left_posterior_communicating_artery,right_posterior_communicating_artery,basilar_tip,other_posterior_circulation,aneurysm_present,image_file
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.56949904638593632206...,MRA,1,-100.0,-100.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.2.826.0.1.3680043.8.498.10004044428023505108...
1,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.12396711188070994245...,MRA,2,-100.0,-100.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.2.826.0.1.3680043.8.498.10004044428023505108...
2,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.27571397853195038984...,MRA,3,-100.0,-100.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.2.826.0.1.3680043.8.498.10004044428023505108...
3,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.60143101667068651693...,MRA,4,-100.0,-100.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.2.826.0.1.3680043.8.498.10004044428023505108...
4,1.2.826.0.1.3680043.8.498.10004044428023505108...,1.2.826.0.1.3680043.8.498.45662927574100362473...,MRA,5,-100.0,-100.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.2.826.0.1.3680043.8.498.10004044428023505108...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1028806,1.2.826.0.1.3680043.8.498.99985209798463601651...,1.2.826.0.1.3680043.8.498.45892616470371688520...,CTA,160,1.0,-1024.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.99985209798463601651...
1028807,1.2.826.0.1.3680043.8.498.99985209798463601651...,1.2.826.0.1.3680043.8.498.50498001331832140551...,CTA,161,1.0,-1024.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.99985209798463601651...
1028808,1.2.826.0.1.3680043.8.498.99985209798463601651...,1.2.826.0.1.3680043.8.498.53285887723729482043...,CTA,162,1.0,-1024.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.99985209798463601651...
1028809,1.2.826.0.1.3680043.8.498.99985209798463601651...,1.2.826.0.1.3680043.8.498.99869751254453760418...,CTA,163,1.0,-1024.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.99985209798463601651...


In [4]:
cols = train_sampled_df.columns[6:6+13]
idx_to_col = {i: x for i, x in enumerate(cols)}
col_to_idx = {x: i for i, x in enumerate(cols)}
cols

Index(['left_infraclinoid_internal_carotid_artery',
       'right_infraclinoid_internal_carotid_artery',
       'left_supraclinoid_internal_carotid_artery',
       'right_supraclinoid_internal_carotid_artery',
       'left_middle_cerebral_artery', 'right_middle_cerebral_artery',
       'anterior_communicating_artery', 'left_anterior_cerebral_artery',
       'right_anterior_cerebral_artery', 'left_posterior_communicating_artery',
       'right_posterior_communicating_artery', 'basilar_tip',
       'other_posterior_circulation'],
      dtype='object')

In [5]:
# result_root_path = '../_results/00_02_09_apply_cropped_reference_setting/predict/20260215_222705'
result_root_path = '../_results/00_02_10_confirm_classification_setting/predict/20260216_192615'

predicts = np.load(os.path.join(result_root_path, 'predict_00.npy'))
predicts.shape, predicts

((216084, 14),
 array([[8.941e-06, 1.645e-05, 8.941e-07, ..., 1.252e-06, 1.818e-05,
         1.568e-05],
        [1.454e-04, 1.881e-04, 2.545e-05, ..., 2.247e-05, 2.673e-04,
         8.798e-04],
        [2.354e-05, 3.195e-05, 5.722e-06, ..., 6.020e-06, 1.388e-04,
         4.506e-05],
        ...,
        [1.609e-06, 1.371e-06, 2.265e-06, ..., 2.205e-06, 2.384e-06,
         3.564e-05],
        [2.301e-05, 6.354e-05, 1.726e-04, ..., 6.289e-04, 4.916e-04,
         5.759e-02],
        [9.537e-07, 1.252e-06, 1.371e-06, ..., 2.325e-06, 5.186e-06,
         1.329e-05]], dtype=float16))

In [6]:
result_root_paths = [
    result_root_path,
]
predicts, labels, uids = [], [], []

NUM_FOLD = 1
for fold_index in range(NUM_FOLD):
    fold_predicts = np.load(os.path.join(result_root_paths[fold_index], f'predict_{fold_index:02d}.npy'))
    fold_labels = np.load(os.path.join(result_root_paths[fold_index], f'label_{fold_index:02d}.npy'))
    fold_uids = np.load(os.path.join(result_root_paths[fold_index], f'uid_{fold_index:02d}.npy'))

    predicts.extend(fold_predicts)
    labels.extend(fold_labels)
    uids.extend(fold_uids)

predicts, labels, uids = np.array(predicts), np.array(labels), np.array(uids)
predicts.shape, labels.shape, uids.shape

((216084, 14), (216084, 14), (216084,))

In [7]:
series_groups = np.array([uid.split('_')[0] for uid in uids])

group_indices = defaultdict(list)
for slide_index, group_name in enumerate(series_groups):
    group_indices[group_name].append(slide_index)

len(group_indices)

876

In [8]:
series_predicts, series_labels, series_uids = [], [], []

for series_uid in np.unique(series_groups):
    group_predicts = predicts[group_indices[series_uid]]

    series_predicts.append(group_predicts.max(0))
    series_labels.append(labels[group_indices[series_uid]].max(0))
    series_uids.append(series_uid)

series_predicts, series_labels, series_uids = np.array(series_predicts), np.array(series_labels), np.array(series_uids)
series_predicts.shape, series_labels.shape, series_uids.shape

((876, 14), (876, 14), (876,))

In [9]:
classes_to_score = {}
scores = []
for index, class_name in enumerate(cols.tolist() + ['aneurysm_present']):
    score = roc_auc_score(series_labels[:, index], series_predicts[:, index])
    classes_to_score[class_name] = score
    scores.append(score)

total_score = (np.mean(scores[:len(scores) - 1]) + scores[-1]) / 2

print(f'Patient-Level AUC: {total_score}')
pprint(classes_to_score)

Patient-Level AUC: 0.7258555664766657
{'aneurysm_present': 0.7358955845893301,
 'anterior_communicating_artery': 0.7985694547415245,
 'basilar_tip': 0.790967939242571,
 'left_anterior_cerebral_artery': 0.6518475750577367,
 'left_infraclinoid_internal_carotid_artery': 0.5665697674418604,
 'left_middle_cerebral_artery': 0.742088648204707,
 'left_posterior_communicating_artery': 0.7361501061425735,
 'left_supraclinoid_internal_carotid_artery': 0.6380458646200395,
 'other_posterior_circulation': 0.6688461538461539,
 'right_anterior_cerebral_artery': 0.6959876543209876,
 'right_infraclinoid_internal_carotid_artery': 0.7028621495327102,
 'right_middle_cerebral_artery': 0.8501391113734086,
 'right_posterior_communicating_artery': 0.8196881091617934,
 'right_supraclinoid_internal_carotid_artery': 0.6438395950459516}


In [11]:
positive_predicts_df = pd.DataFrame(predicts[labels[:, -1] == 1.])
positive_predicts_df.shape, positive_predicts_df.head()

((100120, 14),
              0         1         2         3         4         5         6   \
 0  6.164551e-02  0.086304  0.215820  0.187866  0.000732  0.000269  0.000040   
 1  3.397465e-06  0.000016  0.001980  0.042023  0.040375  0.970215  0.020172   
 2  2.711487e-02  0.179932  0.021652  0.008781  0.000764  0.000035  0.000053   
 3  2.980232e-07  0.000002  0.000343  0.001897  0.112183  0.002726  0.865723   
 4  7.095337e-04  0.001048  0.047150  0.011597  0.028015  0.011780  0.004963   
 
          7         8         9         10        11        12        13  
 0  0.000024  0.000247  0.025421  0.017715  0.001484  0.003124  0.936523  
 1  0.002632  0.002350  0.001346  0.011467  0.011200  0.000897  0.989746  
 2  0.000003  0.000025  0.001037  0.001289  0.003580  0.161743  0.897461  
 3  0.004646  0.006390  0.000376  0.000509  0.003088  0.000173  0.969238  
 4  0.000443  0.001121  0.007725  0.009598  0.008217  0.001741  0.411621  )

In [12]:
positive_predicts_df.describe()

d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: Runt

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
count,1.001200e+05,100120.000000,1.001200e+05,100120.000000,100120.000000,100120.000000,100120.000000,1.001200e+05,100120.000000,100120.000000,1.001200e+05,100120.000000,100120.000000,100120.000000
mean,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
std,8.682251e-03,0.010666,2.458191e-02,0.018906,0.021500,0.035248,0.037598,2.115250e-03,0.002342,0.008614,8.186340e-03,0.009499,0.004234,0.117737
min,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,9.536743e-07,0.000001,8.344650e-07,0.000002,0.000002,0.000002,0.000002,8.344650e-07,0.000002,0.000001,8.940697e-07,0.000001,0.000005,0.000003
50%,4.947186e-06,0.000006,5.900860e-06,0.000011,0.000011,0.000010,0.000012,8.165836e-06,0.000015,0.000007,5.602837e-06,0.000008,0.000021,0.000026
75%,2.586842e-05,0.000037,3.880262e-05,0.000063,0.000067,0.000063,0.000064,7.140636e-05,0.000102,0.000037,3.296137e-05,0.000047,0.000116,0.000267
max,6.743164e-01,0.795898,9.331055e-01,0.738281,0.973145,0.998047,0.998535,1.279297e-01,0.121399,0.488525,3.491211e-01,0.673340,0.217041,1.000000


In [13]:
desc_percentiles = np.arange(start=95, stop=100, step=1) / 100
positive_predicts_df.describe(percentiles=desc_percentiles.tolist())

d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
d:\ProgramData\anaconda3\envs\machine\Lib\site-packages\pandas\core\nanops.py:1487: Runt

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
count,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000,100120.000000
mean,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
std,0.008682,0.010666,0.024582,0.018906,0.021500,0.035248,0.037598,0.002115,0.002342,0.008614,0.008186,0.009499,0.004234,0.117737
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000005,0.000006,0.000006,0.000011,0.000011,0.000010,0.000012,0.000008,0.000015,0.000007,0.000006,0.000008,0.000021,0.000026
95%,0.000509,0.000826,0.001973,0.002090,0.002342,0.002028,0.001285,0.001041,0.001372,0.000911,0.001099,0.001202,0.001965,0.032471
96%,0.000789,0.001240,0.003338,0.003469,0.003708,0.003235,0.001995,0.001351,0.001755,0.001450,0.001748,0.001777,0.002653,0.062133
97%,0.001353,0.001972,0.006449,0.006538,0.006439,0.005619,0.003456,0.001861,0.002405,0.002531,0.003153,0.002993,0.003736,0.147322
98%,0.002867,0.004101,0.014369,0.014732,0.013168,0.011826,0.007637,0.002841,0.003673,0.005612,0.006905,0.005775,0.005959,0.404170
99%,0.009232,0.012192,0.044342,0.044434,0.035889,0.038452,0.027518,0.005535,0.007038,0.015658,0.020005,0.015488,0.011597,0.860747


In [15]:
predict_sum_scores = predicts[labels[:, -1] == 1.].sum(1)
type(predict_sum_scores), predict_sum_scores.shape

(numpy.ndarray, (100120,))

In [16]:
POSITIVE_THRESHOLD = 1.6
positive_uids = uids[labels[:, -1] == 1.][predict_sum_scores > POSITIVE_THRESHOLD]
len(positive_uids), positive_uids

(729,
 array(['1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831_I_454',
        '1.2.826.0.1.3680043.8.498.10034081836061566510187499603024895557_I_47',
        '1.2.826.0.1.3680043.8.498.10035643165968342618460849823699311381_I_78',
        '1.2.826.0.1.3680043.8.498.10076056930521523789588901704956188485_I_75',
        '1.2.826.0.1.3680043.8.498.10143240284902513794767720489625125957_I_133',
        '1.2.826.0.1.3680043.8.498.10161806953566875622930260306554507426_I_54',
        '1.2.826.0.1.3680043.8.498.10327401654089434788594119044276508319_I_155',
        '1.2.826.0.1.3680043.8.498.10651378641908724856730013035772912257_I_596',
        '1.2.826.0.1.3680043.8.498.10722329050491929401656671952575354429_I_93',
        '1.2.826.0.1.3680043.8.498.10743199796364362163988736837321335182_I_119',
        '1.2.826.0.1.3680043.8.498.10743199796364362163988736837321335182_I_136',
        '1.2.826.0.1.3680043.8.498.10834923084007253548108699309528531373_I_86',
        '1.2.826

In [17]:
np.unique([positive_uid.split('_')[0] for positive_uid in positive_uids]).shape

(175,)

In [19]:
positive_data_rows = []
for positive_uid in positive_uids:
    
    row_index = filename_to_index[positive_uid]

    positive_data_rows.append(train_sampled_df.iloc[row_index])

positive_data_df = pd.DataFrame(positive_data_rows)
positive_data_df

,SeriesUID,InstanceUID,Modality,InstanceNumber,RescaleSlope,RescaleIntercept,left_infraclinoid_internal_carotid_artery,right_infraclinoid_internal_carotid_artery,left_supraclinoid_internal_carotid_artery,right_supraclinoid_internal_carotid_artery,...,right_middle_cerebral_artery,anterior_communicating_artery,left_anterior_cerebral_artery,right_anterior_cerebral_artery,left_posterior_communicating_artery,right_posterior_communicating_artery,basilar_tip,other_posterior_circulation,aneurysm_present,image_file
2711,1.2.826.0.1.3680043.8.498.10022796280698534221...,1.2.826.0.1.3680043.8.498.53868409774237283281...,CTA,454,-100.0,-100.0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.10022796280698534221...
3656,1.2.826.0.1.3680043.8.498.10034081836061566510...,1.2.826.0.1.3680043.8.498.71237104731452368587...,CTA,47,1.0,0.0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.10034081836061566510...
3770,1.2.826.0.1.3680043.8.498.10035643165968342618...,1.2.826.0.1.3680043.8.498.46752468449107005352...,CTA,78,-100.0,-100.0,0,0,0,1,...,0,0,0,1,0,0,0,0,1,1.2.826.0.1.3680043.8.498.10035643165968342618...
7200,1.2.826.0.1.3680043.8.498.10076056930521523789...,1.2.826.0.1.3680043.8.498.81558459046210612235...,MRA,75,-100.0,-100.0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,1.2.826.0.1.3680043.8.498.10076056930521523789...
11887,1.2.826.0.1.3680043.8.498.10143240284902513794...,1.2.826.0.1.3680043.8.498.77238922582709988246...,CTA,133,1.0,0.0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1.2.826.0.1.3680043.8.498.10143240284902513794...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020578,1.2.826.0.1.3680043.8.498.99171153540341946985...,1.2.826.0.1.3680043.8.498.72646082588296120638...,CTA,118,1.0,-1024.0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,1.2.826.0.1.3680043.8.498.99171153540341946985...
1020579,1.2.826.0.1.3680043.8.498.99171153540341946985...,1.2.826.0.1.3680043.8.498.10563879224733419350...,CTA,119,1.0,-1024.0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,1.2.826.0.1.3680043.8.498.99171153540341946985...
1020582,1.2.826.0.1.3680043.8.498.99171153540341946985...,1.2.826.0.1.3680043.8.498.47411142735106315085...,CTA,122,1.0,-1024.0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,1.2.826.0.1.3680043.8.498.99171153540341946985...
1020583,1.2.826.0.1.3680043.8.498.99171153540341946985...,1.2.826.0.1.3680043.8.498.25472954612188547145...,CTA,123,1.0,-1024.0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,1.2.826.0.1.3680043.8.498.99171153540341946985...


In [20]:
positive_data_df.to_csv('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/train_sampled_positive.csv', index=False)